# Create TUBITAK Awards from TRDizin Projects

Creates OpenAlex award rows from ULAKBIM/TRDizin's official TUBITAK-funded completed-project database.

- Funder: Turkiye Bilimsel ve Teknolojik Arastirma Kurumu (`F4320322626`; ROR `https://ror.org/04w9kkr77`; DOI `10.13039/501100004410`).
- Source authority: TUBITAK's own announcement for the ULAKBIM TUBITAK-supported projects database, plus TRDizin's public project-search API.
- Source method: official source ladder 3, public search/index API using `facet-documentType=PROJECT`.
- Local validation on 2026-06-02: 27,828 raw PROJECT records; 27,309 unique native project-number award rows after dropping 302 records without project numbers and collapsing 206 duplicate project-number groups.
- Coverage: 100% native project number/title/project group/year/subject tags/report pages; 98.9% start dates; 99.98% end dates after nulling `1900-01-01` placeholders; 27,297/27,309 lead names; 34.2% lead ORCID; 94.4% PDF identifiers.
- Amount/currency decision: NULL by source authority. TRDizin project records do not publish per-project award amounts or currencies, so Step 6.7 is waived like other official no-amount project/fellowship ingests.
- Institution/country decision: NULL by source authority. The API contributor institution fields are null in project records; country is not inferred from TUBITAK or project-group context.
- Provenance: `trdizin_tubitak_projects`; priority: `200`.


## Imports / Config


In [ ]:
FUNDER_ID = 4320322626
PROVENANCE = "trdizin_tubitak_projects"
PRIORITY = 200


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.tubitak_trdizin_projects_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/tubitak/tubitak_trdizin_projects.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-06-02 found 27,309 rows.
SELECT COUNT(*) AS total_rows
FROM openalex.awards.tubitak_trdizin_projects_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.tubitak_trdizin_projects_raw;


In [ ]:
%sql
SELECT
    funder_award_id,
    trdizin_id,
    display_name,
    project_group,
    publication_year,
    start_date,
    end_date,
    lead_name,
    lead_role,
    lead_orcid,
    landing_page_url
FROM openalex.awards.tubitak_trdizin_projects_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Native Keys, Dates, and Contributor Coverage


In [ ]:
%sql
-- Money/date-field scan per runbook Step 1.5. Amount columns are absent by source authority.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'tubitak_trdizin_projects_raw'
  AND LOWER(column_name) RLIKE 'amount|currency|year|date|fund|grant|award|project|orcid|lead|contributor|subject'
ORDER BY column_name;


In [ ]:
%sql
-- Coverage profile from the official TRDizin project API.
SELECT
    COUNT(*) AS total_rows,
    COUNT(funder_award_id) AS native_id_rows,
    ROUND(try_divide(COUNT(funder_award_id), COUNT(*)) * 100.0, 1) AS pct_native_id,
    COUNT(display_name) AS title_rows,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    COUNT(description) AS description_rows,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    COUNT(project_group) AS project_group_rows,
    ROUND(try_divide(COUNT(project_group), COUNT(*)) * 100.0, 1) AS pct_project_group,
    COUNT(publication_year) AS publication_year_rows,
    ROUND(try_divide(COUNT(publication_year), COUNT(*)) * 100.0, 1) AS pct_publication_year,
    COUNT(start_date) AS start_date_rows,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    COUNT(end_date) AS end_date_rows,
    ROUND(try_divide(COUNT(end_date), COUNT(*)) * 100.0, 1) AS pct_end_date,
    COUNT(lead_name) AS lead_name_rows,
    ROUND(try_divide(COUNT(lead_name), COUNT(*)) * 100.0, 1) AS pct_lead_name,
    COUNT(lead_orcid) AS lead_orcid_rows,
    ROUND(try_divide(COUNT(lead_orcid), COUNT(*)) * 100.0, 1) AS pct_lead_orcid,
    COUNT(pdf_id) AS pdf_rows,
    ROUND(try_divide(COUNT(pdf_id), COUNT(*)) * 100.0, 1) AS pct_pdf
FROM openalex.awards.tubitak_trdizin_projects_raw;


In [ ]:
%sql
-- Native-key inspection. funder_award_id is the official TUBITAK project number.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT trdizin_id) AS distinct_trdizin_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    SUM(CASE WHEN TRY_CAST(duplicate_source_count AS INT) > 1 THEN 1 ELSE 0 END) AS rows_collapsed_from_duplicate_source_ids
FROM openalex.awards.tubitak_trdizin_projects_raw;


In [ ]:
%sql
-- Must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.tubitak_trdizin_projects_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC, funder_award_id;


In [ ]:
%sql
-- Project-group distribution.
SELECT project_group, COUNT(*) AS rows
FROM openalex.awards.tubitak_trdizin_projects_raw
GROUP BY project_group
ORDER BY rows DESC
LIMIT 50;


In [ ]:
%sql
-- Year and placeholder-date inspection. publication_year=1900 is a source placeholder and is nulled in Step 2.
SELECT
    MIN(TRY_CAST(publication_year AS INT)) AS min_publication_year,
    MAX(TRY_CAST(publication_year AS INT)) AS max_publication_year,
    MIN(TRY_TO_DATE(start_date, 'yyyy-MM-dd')) AS min_start_date,
    MAX(TRY_TO_DATE(start_date, 'yyyy-MM-dd')) AS max_start_date,
    MIN(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) AS min_end_date,
    MAX(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) AS max_end_date,
    SUM(CASE WHEN publication_year = '1900' THEN 1 ELSE 0 END) AS publication_year_1900_rows,
    SUM(CASE WHEN end_date = '1900-01-01' THEN 1 ELSE 0 END) AS end_date_1900_rows
FROM openalex.awards.tubitak_trdizin_projects_raw;


## Step 1.6: Funder Existence Fail-Fast


In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for TUBITAK F4320322626'
) AS funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320322626;


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320322626;


## Step 2: Transform to OpenAlex Award Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.tubitak_awards
USING delta
AS
WITH
tubitak_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322626
),
raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        CASE
            WHEN TRY_CAST(publication_year AS INT) = 1900 THEN NULL
            WHEN TRY_CAST(publication_year AS INT) > YEAR(current_date()) + 1 THEN NULL
            ELSE TRY_CAST(publication_year AS INT)
        END AS parsed_publication_year,
        FROM_JSON(other_investigators_json, 'ARRAY<STRUCT<name:STRING,given_name:STRING,family_name:STRING,orcid:STRING,duty:STRING,order:STRING,role_start:STRING,institution_name:STRING,institution_title:STRING,institution_root_title:STRING,institution_country:STRING>>') AS other_contributors
    FROM openalex.awards.tubitak_trdizin_projects_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS id,
        TRIM(g.display_name) AS display_name,
        CASE WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL ELSE TRIM(g.description) END AS description,
        f.funder_id,
        TRIM(g.funder_award_id) AS funder_award_id,
        CAST(NULL AS DOUBLE) AS amount,
        CAST(NULL AS STRING) AS currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'research' AS funding_type,
        COALESCE(NULLIF(TRIM(g.project_group), ''), 'TRDizin PROJECT') AS funder_scheme,
        'trdizin_tubitak_projects' AS provenance,
        g.parsed_start_date AS start_date,
        g.parsed_end_date AS end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_publication_year) AS start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_publication_year) AS end_year,
        CASE
            WHEN g.lead_name IS NULL OR TRIM(g.lead_name) = '' THEN CAST(NULL AS STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >)
            ELSE struct(
                NULLIF(TRIM(g.lead_given_name), '') AS given_name,
                NULLIF(TRIM(g.lead_family_name), '') AS family_name,
                NULLIF(TRIM(g.lead_orcid), '') AS orcid,
                g.parsed_start_date AS role_start,
                struct(
                    CAST(NULL AS STRING) AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        END AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CASE
            WHEN g.other_contributors IS NULL OR SIZE(g.other_contributors) = 0 THEN CAST(NULL AS ARRAY<STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >>)
            ELSE TRANSFORM(g.other_contributors, c -> struct(
                NULLIF(TRIM(c.given_name), '') AS given_name,
                NULLIF(TRIM(c.family_name), '') AS family_name,
                NULLIF(TRIM(c.orcid), '') AS orcid,
                g.parsed_start_date AS role_start,
                struct(
                    CAST(NULL AS STRING) AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            ))
        END AS investigators,
        NULLIF(TRIM(g.landing_page_url), '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM raw_prepared g
    CROSS JOIN tubitak_funder f
)
SELECT * FROM awards_transformed;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_native_ids,
    COUNT(amount) AS amount_rows
FROM openalex.awards.tubitak_awards;


In [ ]:
%sql
SELECT
    id,
    display_name,
    funder_award_id,
    funding_type,
    funder_scheme,
    start_year,
    end_year,
    lead_investigator.given_name AS lead_given_name,
    lead_investigator.family_name AS lead_family_name,
    lead_investigator.orcid AS lead_orcid,
    SIZE(investigators) AS investigator_count
FROM openalex.awards.tubitak_awards
LIMIT 20;


## Step 3: Delete Existing Rows and Insert Fresh Priority 200 Rows


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trdizin_tubitak_projects' AND priority = 200;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    200 AS priority
FROM openalex.awards.tubitak_awards;


In [ ]:
%sql
SELECT
    provenance,
    priority,
    COUNT(*) AS rows,
    COUNT(DISTINCT funder_award_id) AS distinct_native_ids,
    COUNT(DISTINCT id) AS distinct_openalex_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trdizin_tubitak_projects'
GROUP BY provenance, priority;


## Step 6: Verification Queries


In [ ]:
%sql
-- Duplicate OpenAlex IDs within this ingest: must return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trdizin_tubitak_projects' AND priority = 200
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY rows DESC, id;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS title_rows,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    COUNT(description) AS description_rows,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    COUNT(funder_award_id) AS native_id_rows,
    ROUND(try_divide(COUNT(funder_award_id), COUNT(*)) * 100.0, 1) AS pct_native_id,
    COUNT(amount) AS amount_rows,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS currency_rows,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_currency,
    COUNT(start_year) AS start_year_rows,
    ROUND(try_divide(COUNT(start_year), COUNT(*)) * 100.0, 1) AS pct_start_year,
    COUNT(end_year) AS end_year_rows,
    ROUND(try_divide(COUNT(end_year), COUNT(*)) * 100.0, 1) AS pct_end_year,
    COUNT(lead_investigator.family_name) AS lead_rows,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) AS pct_lead,
    COUNT(lead_investigator.orcid) AS lead_orcid_rows,
    ROUND(try_divide(COUNT(lead_investigator.orcid), COUNT(*)) * 100.0, 1) AS pct_lead_orcid
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trdizin_tubitak_projects' AND priority = 200;


In [ ]:
%sql
-- Amount/currency are intentionally NULL: TRDizin does not publish per-project money.
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount,
    'WAIVED: TRDizin PROJECT records do not publish per-project award amounts or currencies.' AS amount_decision
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trdizin_tubitak_projects' AND priority = 200;


In [ ]:
%sql
SELECT
    funder_id,
    funder.display_name AS funder_name,
    provenance,
    priority,
    COUNT(*) AS rows,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trdizin_tubitak_projects' AND priority = 200
GROUP BY funder_id, funder.display_name, provenance, priority;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trdizin_tubitak_projects' AND priority = 200
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 40;


In [ ]:
%sql
-- Must be zero after source-year placeholder handling and central future-year guard.
SELECT COUNT(*) AS future_start_year_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trdizin_tubitak_projects'
  AND priority = 200
  AND start_year > YEAR(current_date()) + 1;


In [ ]:
%sql
SELECT
    id,
    display_name,
    description,
    funder_award_id,
    amount,
    currency,
    funding_type,
    funder_scheme,
    start_date,
    end_date,
    lead_investigator,
    investigators,
    landing_page_url,
    works_api_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trdizin_tubitak_projects' AND priority = 200
ORDER BY funder_award_id
LIMIT 25;


## Handoff / Admin Notes

Contractor-complete handoff only. The script and notebook are ready, but the contractor has no S3 or Databricks access.

Admin must:

- Run `scripts/local/tubitak_to_s3.py` without `--skip-upload` to upload `tubitak_trdizin_projects.parquet`.
- Run this Databricks notebook.
- Inspect the Step 6 verification cells, especially the no-amount waiver, placeholder-year handling, and duplicate-native-ID collapse.
- Run `CreateAwards.ipynb` after QA.
- Flip the tracker row from Step 4 to Complete only after production verification.
